In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from astropy.io import fits
import glob
from datetime import datetime, timedelta
from interpolation import interp1d
from utils import *
import sunpy.visualization.colormaps as cm

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
def show(data):
    x_lims = mdates.date2num([datetime(2024,1,1), datetime(2026,1,1)])
    y_lims = [-1, 1]

    fig, ax = plt.subplots(figsize=(14,9))
    ax.imshow(data, cmap='seismic', vmin=-10, vmax=10, interpolation='nearest',
              extent=(x_lims[0], x_lims[1], y_lims[0], y_lims[1]), aspect='auto', origin='lower')

    ax.set_yticks(np.sin(np.arange(-90, 91, 15) * np.pi / 180), np.arange(-90, 91, 15))

    ax.xaxis_date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y/%m'))

    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

In [10]:
files_hmi = np.array(sorted(glob.glob('/home/ulyanov/data/sdo/hmi/synoptic_maps/*')))

data_hmi = []
dates_hmi = []

for file in files_hmi:
    with fits.open(file) as hdul:
        header = hdul[0].header.copy()
        data = hdul[0].data.copy()

    data = rebin(data, 2)
    data_hmi.append(data)

    date = datetime.fromisoformat(header['T_OBS'][:-4].replace('.', '').replace('_', 'T'))
    dates_hmi.append(date)

data_hmi = np.array(data_hmi)
dates_hmi = np.array(dates_hmi)

flux_hmi = np.nanmean(data_hmi, axis=-1)

/tmp/ipykernel_73253/687622684.py:20: RuntimeWarning: Mean of empty slice
  flux_hmi = np.nanmean(data_hmi, axis=-1)


In [18]:
files_fdt = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/synoptic_maps/*')))

data_fdt = []
weights = []
dates_fdt = []

for file in files_fdt:
    s = np.load(file)
    mean = s['mean']
    weight = s['weight']

    start = datetime.fromisoformat(str(s['start']))
    end = datetime.fromisoformat(str(s['end']))
    delta = end - start
    date = start + delta / 2

    data_fdt.append(mean)
    weights.append(weight)
    dates_fdt.append(date)

data_fdt = np.array(data_fdt)
weights = np.array(weights)
dates_fdt = np.array(dates_fdt)

flux_fdt = np.nanmean(data_fdt, axis=-1)

/tmp/ipykernel_73253/3207815866.py:25: RuntimeWarning: Mean of empty slice
  flux_fdt = np.nanmean(data_fdt, axis=-1)


In [12]:
fig, ax = plt.subplots(figsize=(14,8))
ax.imshow(data_fdt[3], aspect='auto', origin='lower', cmap='hmimag',
           extent=(0, 360, -1, 1), vmin=-1000, vmax=1000)
ax.set_yticks(np.sin(np.arange(-90, 91, 15) * np.pi / 180), np.arange(-90, 91, 15))
plt.tight_layout()

In [22]:
show(rebin(flux_hmi, 8, axis=1).T)

In [21]:
show(rebin(flux_fdt, 8, axis=1).T)

In [7]:
dates_fdt[3]

datetime.datetime(2024, 4, 1, 1, 37, 36)

In [20]:
dsine = 1 / 359.5
lat = np.arange(-1,1 + dsine / 2, dsine)
lat = np.arcsin(lat.clip(-1,1)) * 180 / np.pi

plt.figure(figsize=(10,8))
plt.plot(lat, flux_fdt[3], label='FDT')
plt.plot(lat, flux_hmi[3], label='HMI')

plt.xlim(-90,90)
plt.ylim(-10,10)

plt.title('CR 2290/2291, North pole')
plt.xlabel('Longitude, degrees')
plt.ylabel(r'Flux density, Mx / cm$^2$')
plt.legend()
plt.grid(True)
plt.tight_layout()